# CM32022/52035 Advanced Computer Vision: Coursework 1

**Group XX:** TODO member names

### How to run (for markers)
1. Set up the environment described in the README (TODO: write the README, listing packages and versions).
2. Put the dataset files where the **Config** cell expects them. All paths are relative to this notebook.
3. To evaluate the submitted models on a different test set, point `TEST_PATH` (and the task 5 paths) at the new files, set `RUN_TRAINING = False`, and run all cells. Trained weights are loaded from `models/`.

### Submission checklist
- [ ] `Group XX report.pdf`: main part at most 9,000 words (references and appendices excluded)
- [ ] `Group XX GCF.pdf`
- [ ] `Group XX code.zip`: this notebook, `models/` and a README with the environment setup; no absolute paths
- [ ] Major steps of the code commented
- [ ] GenAI use acknowledged in the report (brief, section 7.2.1)

### GenAI use log
Keep this up to date for the acknowledgement in the report.
- Notebook scaffold (section structure, config, seeding/checkpoint/results helpers and empty per-task stubs) generated with Kiro, an AI assistant, on TODO date.
- TODO: any further use.

## Setup

In [ ]:
# Optional, Google Colab only: mount Drive and move into the project folder so the relative
# paths in the Config cell work. Leave this commented out when running locally.
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive/TODO-project-folder

In [ ]:
# Tasks 1-7 may only use PyTorch, NumPy and Matplotlib (plus the Python standard library).
# The extra libraries allowed in task 6 (t-SNE) and task 8 (SIFT, SVM, K-means, Fisher Vectors)
# are imported in those sections.
# torchvision: the brief bans torchvision.datasets.CIFAR100 specifically. If you want other parts
# of torchvision (e.g. transforms), confirm on the Moodle FAQ that it counts as PyTorch.
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
# ---------------- Config ----------------
# Paths are relative to this notebook so the code runs on any machine (no absolute paths allowed).
DATA_DIR = Path("data")        # TODO: where you put the dataset files provided on Moodle
MODEL_DIR = Path("models")     # trained weights; include this folder in the code zip
RESULTS_DIR = Path("results")  # metrics (results.json) and figures for the report
FIG_DIR = RESULTS_DIR / "figures"
for d in (MODEL_DIR, RESULTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# TODO: replace the placeholders once you know the names and layout of the provided files.
TRAIN_PATH = DATA_DIR / "TODO"        # main training set: 100 images per fine class (10,000)
TEST_PATH = DATA_DIR / "TODO"         # main test set: 25 per fine class (2,500); markers swap in a private one
TASK5_TRAIN_PATH = DATA_DIR / "TODO"  # task 5 training split (20 fine classes removed)
TASK5_TEST_PATH = DATA_DIR / "TODO"   # task 5 test split
TASK5_SUPPORT_PATHS = {               # support-set size -> path, e.g. {5: DATA_DIR / "TODO"}
}

# True: train every model. False: load the saved weights from MODEL_DIR and only evaluate
# (use this to re-run the evaluation on a new test set).
RUN_TRAINING = True

SEED = 0
NUM_FINE = 100
NUM_COARSE = 20

# DataLoader worker processes. Keep 0 in a notebook on macOS/Windows: worker processes there cannot
# import classes defined in the notebook. On Colab (Linux), 2 or more speeds up loading.
NUM_WORKERS = 0

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon GPU
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

In [ ]:
# ---------------- Helpers: seeding, config checks, loaders, checkpoints ----------------
def set_seed(seed: int = SEED) -> None:
    """Seed Python, NumPy and PyTorch (CPU and GPU) so runs are repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def check_cfg(cfg: dict) -> None:
    """Stop early if a config dict still has entries left as None or empty."""
    missing = [k for k, v in cfg.items()
               if v is None or (isinstance(v, (list, tuple, dict)) and len(v) == 0)]
    if missing:
        raise ValueError(f"Fill in these config entries first: {missing}")


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    """DataLoader with the notebook-wide worker settings. Use shuffle=False for evaluation."""
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")


def save_checkpoint(model: nn.Module, name: str, **extra) -> None:
    """Save the weights to MODEL_DIR/name, plus any extra info (e.g. cfg=...).

    Keep `extra` to plain Python types (numbers, strings, lists, dicts) so that it loads with
    torch.load's default weights_only=True.
    """
    torch.save({"state_dict": model.state_dict(), **extra}, MODEL_DIR / name)


def load_checkpoint(model: nn.Module, name: str) -> dict:
    """Load weights saved by save_checkpoint into `model` (in place) and return the extra info."""
    ckpt = torch.load(MODEL_DIR / name, map_location="cpu")
    model.load_state_dict(ckpt.pop("state_dict"))
    return ckpt

In [ ]:
# ---------------- Helpers: results and figures ----------------
RESULTS = {}  # run name -> metrics; written to results/results.json by save_results()


def save_results() -> None:
    """Write RESULTS to results/results.json. Store scalars or lists; numbers are converted with float()."""
    (RESULTS_DIR / "results.json").write_text(json.dumps(RESULTS, indent=2, default=float))


def savefig(fig: plt.Figure, name: str, dpi: int = 200) -> None:
    """Save a figure for the report as results/figures/<name>.png."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight")


def plot_history(history: dict, title: str = "") -> plt.Figure:
    """One subplot per metric in `history` (metric name -> list of per-epoch values)."""
    fig, axes = plt.subplots(1, len(history), figsize=(4 * len(history), 3), squeeze=False)
    for ax, (metric, values) in zip(axes[0], history.items()):
        ax.plot(range(1, len(values) + 1), values)
        ax.set_xlabel("epoch")
        ax.set_title(metric)
    fig.suptitle(title)
    fig.tight_layout()
    return fig

## Data

The brief forbids `torchvision.datasets.CIFAR100`, so the loading code and the `Dataset` class are yours. `load_split` returns raw `uint8` images so the same loader serves the CNN tasks and the SIFT task (task 8).

In [ ]:
def load_split(path: Path):
    """Load one split of the provided data.

    Returns
    -------
    images : np.ndarray of uint8, shape (N, 32, 32, 3)
    fine_labels : np.ndarray of int64, shape (N,)
    coarse_labels : np.ndarray of int64, shape (N,)
    """
    # TODO: implement for the file format provided on Moodle. If the files include class names,
    # keep them too (useful for plot legends and the report).
    raise NotImplementedError


class CIFAR100Subset(Dataset):
    """Your own Dataset class. Each item is (image_tensor, fine_label, coarse_label)."""

    def __init__(self, images: np.ndarray, fine_labels: np.ndarray, coarse_labels: np.ndarray,
                 transform=None):
        # TODO
        raise NotImplementedError

    def __len__(self) -> int:
        # TODO
        raise NotImplementedError

    def __getitem__(self, idx: int):
        # TODO: turn the uint8 HWC image into a float CHW tensor, apply self.transform,
        # and return (image, fine_label, coarse_label).
        raise NotImplementedError

In [ ]:
# TODO: preprocessing and augmentation (justify your choices in the report). Things to check:
#  - the input preprocessing the backbone expects from its ImageNet pre-training
#  - test-time preprocessing must be deterministic (no random augmentation)
train_transform = None  # TODO
test_transform = None   # TODO

In [ ]:
train_images, train_fine, train_coarse = load_split(TRAIN_PATH)
test_images, test_fine, test_coarse = load_split(TEST_PATH)

# TODO: build train_set and val_set (CIFAR100Subset objects).
# Tune hyper-parameters on a validation split carved out of the training set, not on the test set:
# marking uses a different, private test set.
train_set = None  # TODO
val_set = None    # TODO

test_set = CIFAR100Subset(test_images, test_fine, test_coarse, transform=test_transform)
# Fixed order, so predictions line up with test_fine / test_coarse.
test_loader = make_loader(test_set, batch_size=256, shuffle=False)

# TODO: sanity check: show a few training images with their fine and coarse labels.

## Backbone

Only the MobileNetV3 backbone provided for the unit, with its ImageNet weights, may be used (no other model or pre-training). Tasks 1-3 fine-tune it together with your classifier; task 4 trains the backbone alone.

In [ ]:
def load_backbone() -> nn.Module:
    """Return a fresh copy of the provided MobileNetV3 backbone (ImageNet weights, no classifier).

    It must map a batch of images (B, 3, H, W) to embeddings (B, D).
    """
    # TODO: load the backbone exactly as provided for the unit.
    raise NotImplementedError


# TODO: pass a dummy batch (preprocessed like your real inputs) through load_backbone() and check
# that the output has shape (B, D). Your classifier heads need D.
EMB_DIM = None  # TODO

## Classification building blocks (tasks 1-3)

Classifiers must be built from `torch.nn` classes. Existing loss functions are allowed.

In [ ]:
class ClassifierHead(nn.Module):
    """Classifier h: embedding (B, D) -> class logits (B, num_classes), built from torch.nn modules.

    Add constructor arguments for the design choices you want to compare (e.g. depth, hidden size,
    dropout) and record them in the task config.
    """

    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError


class ImageClassifier(nn.Module):
    """Backbone f followed by head h, i.e. y_hat = h(f(x)). Fine-tuning updates both."""

    def __init__(self, backbone: nn.Module, head: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x: torch.Tensor):
        return self.head(self.backbone(x))

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, loss_fn, label: str) -> dict:
    """Train for one epoch on the "fine" or "coarse" labels and return epoch metrics, e.g. {"loss": ...}."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> torch.Tensor:
    """Predicted class index for every image in `loader`, in loader order, as a CPU tensor."""
    # TODO (remember model.eval())
    raise NotImplementedError


def top1_accuracy(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Fraction of predictions equal to the ground-truth labels."""
    # TODO
    raise NotImplementedError

## Task 1: Fine-grained classification (8 marks)

Train backbone $f$ and classifier $h$ on $D_{train}$ to predict the fine label, $\hat{y}^f = h(f(x))$. Evaluate top-1 accuracy on $D_{test}$.

**Report:**
- The classifier architecture and how your choices affected performance (number of layers, hidden size, dropout).
- The loss function and optimiser, and why. If you tried several, how did they compare?
- The effect of hyper-parameters such as learning rate and batch size.
- Test-set performance.

In [ ]:
# Final configuration for task 1. Run your hyper-parameter experiments however you like (e.g. loop
# over configs and keep each run's history for the report); record the final choice here.
T1_CFG = {
    "epochs": None,      # TODO
    "batch_size": None,  # TODO
    "lr": None,          # TODO
    # TODO: optimiser, weight decay, head settings, ...
}
check_cfg(T1_CFG)

set_seed(SEED)
t1_model = ImageClassifier(load_backbone(), ClassifierHead(EMB_DIM, NUM_FINE)).to(DEVICE)

if RUN_TRAINING:
    # TODO: optimiser and loss; train for T1_CFG["epochs"] epochs with train_one_epoch on the fine
    # labels, tracking validation accuracy; keep a history dict and plot it with plot_history.
    raise NotImplementedError
    save_checkpoint(t1_model, "task1_fine.pt", cfg=T1_CFG)
else:
    load_checkpoint(t1_model, "task1_fine.pt")

RESULTS["task1"] = {"top1_fine": top1_accuracy(predict(t1_model, test_loader), torch.as_tensor(test_fine))}
print(RESULTS["task1"])

## Task 2: Coarse-grained classification (8 marks)

Same as task 1, but predict the coarse label (20 superclasses), $\hat{y}^c = h(f(x))$.

**Report:** the classifier design choices and test performance, as in task 1.

In [ ]:
T2_CFG = {
    "epochs": None,      # TODO
    "batch_size": None,  # TODO
    "lr": None,          # TODO
    # TODO: optimiser, weight decay, head settings, ...
}
check_cfg(T2_CFG)

set_seed(SEED)
t2_model = ImageClassifier(load_backbone(), ClassifierHead(EMB_DIM, NUM_COARSE)).to(DEVICE)

if RUN_TRAINING:
    # TODO: as task 1, but on the coarse labels.
    raise NotImplementedError
    save_checkpoint(t2_model, "task2_coarse.pt", cfg=T2_CFG)
else:
    load_checkpoint(t2_model, "task2_coarse.pt")

RESULTS["task2"] = {"top1_coarse": top1_accuracy(predict(t2_model, test_loader), torch.as_tensor(test_coarse))}
print(RESULTS["task2"])

## Task 3: Fine- and coarse-grained classification together (10 marks)

One model predicts both labels. Evaluate on $D_{test}$: (i) fine top-1 accuracy, (ii) coarse top-1 accuracy, (iii) exact match (subset accuracy), where an image counts as correct only if both predictions are correct.

**Report:**
- The classifier design choices and the three metrics.
- How the joint model compares with the single-label models from tasks 1 and 2.
- Across tasks 1-3: which was easiest or hardest, which got the best or poorest results, and why. Did you use the same classifier architecture for all three?

In [ ]:
class JointHead(nn.Module):
    """Classifier(s) predicting both labels from one embedding: forward returns (fine_logits, coarse_logits)."""

    def __init__(self, in_dim: int, num_fine: int, num_coarse: int):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, z: torch.Tensor):
        # TODO
        raise NotImplementedError


def joint_loss(fine_logits, coarse_logits, fine_y, coarse_y) -> torch.Tensor:
    """Combine the fine and coarse losses into the scalar you optimise."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def predict_joint(model: nn.Module, loader: DataLoader):
    """Return (fine_pred, coarse_pred) CPU tensors for every image in `loader`, in loader order."""
    # TODO
    raise NotImplementedError


def exact_match(fine_pred, coarse_pred, fine_y, coarse_y) -> float:
    """Fraction of images where both the fine and the coarse prediction are correct."""
    # TODO
    raise NotImplementedError

In [ ]:
T3_CFG = {
    "epochs": None,      # TODO
    "batch_size": None,  # TODO
    "lr": None,          # TODO
    # TODO: optimiser, weight decay, head settings, ...
}
check_cfg(T3_CFG)

set_seed(SEED)
t3_model = ImageClassifier(load_backbone(), JointHead(EMB_DIM, NUM_FINE, NUM_COARSE)).to(DEVICE)

if RUN_TRAINING:
    # TODO: train with joint_loss (train_one_epoch uses a single label, so write a joint version).
    raise NotImplementedError
    save_checkpoint(t3_model, "task3_joint.pt", cfg=T3_CFG)
else:
    load_checkpoint(t3_model, "task3_joint.pt")

t3_fine, t3_coarse = predict_joint(t3_model, test_loader)
y_fine, y_coarse = torch.as_tensor(test_fine), torch.as_tensor(test_coarse)
RESULTS["task3"] = {
    "top1_fine": top1_accuracy(t3_fine, y_fine),
    "top1_coarse": top1_accuracy(t3_coarse, y_coarse),
    "exact_match": exact_match(t3_fine, t3_coarse, y_fine, y_coarse),
}
print(RESULTS["task3"])

## Embedding and k-NN building blocks (tasks 4-8)

In tasks 5, 7 and 8 the k-NN classifier must be your own implementation, in PyTorch or NumPy.

In [ ]:
@torch.no_grad()
def extract_embeddings(backbone: nn.Module, loader: DataLoader) -> torch.Tensor:
    """f(x) for every image in `loader`, in loader order, as an (N, D) CPU tensor.

    Use deterministic preprocessing (test_transform) and shuffle=False, also when embedding
    training or support images.
    """
    # TODO (remember backbone.eval())
    raise NotImplementedError


def pairwise_distances(a: torch.Tensor, b: torch.Tensor, metric: str = "euclidean") -> torch.Tensor:
    """(len(a), len(b)) distance matrix for each metric you compare (e.g. "euclidean", "cosine").

    Keep one convention for every metric: smaller means closer.
    """
    # TODO
    raise NotImplementedError


def knn_predict(query: torch.Tensor, support: torch.Tensor, support_labels: torch.Tensor,
                k: int, metric: str = "euclidean") -> torch.Tensor:
    """Your own k-NN classifier: a predicted label for each query from its k nearest support embeddings."""
    # TODO (also decide how to break ties in the vote)
    raise NotImplementedError

## Task 4: Ranking images with the triplet loss (22 marks)

Train the backbone alone so that $d(f(x_a), f(x_p)) + \alpha < d(f(x_a), f(x_n))$, where positives share the anchor's label and negatives do not. Run two sets of experiments, forming triplets from (i) the fine labels and (ii) the coarse labels. Tune the margin $\alpha$.

The triplet sampling and the triplet loss must be your own (no existing sampling or loss functions).

**Evaluation on $D_{test}$:**
- Recall@k for $k \in \{5, 10, 50, 100\}$. For each test image $x_i$, rank the *other* test images by embedding distance and compute $R(x_i, k) = \frac{1}{|y_i|}\sum_{j=1}^{k} \mathbb{1}[\dot{y}(x_j) = y_i]$, where $|y_i|$ is the class size. Report the mean over the test set.
- The mean distance over all valid (anchor, positive) pairs and over all (anchor, negative) pairs. The positive mean should be smaller.

**Report:**
- Your triplet sampling strategy and why; if you tried several, how they affected results.
- The distance metric and why.
- How performance varies with $\alpha$.
- How the fine- and coarse-label experiments differ.
- The fundamental differences between the triplet loss and the classifier approach of tasks 1-3.

In [ ]:
# ---------------- Triplet sampling (your own implementation) ----------------
# TODO: implement how (anchor, positive, negative) triplets are drawn from D_train, given the labels
# that define positives and negatives (fine or coarse). Structure it however suits your strategy.


def triplet_loss(anchor: torch.Tensor, positive: torch.Tensor, negative: torch.Tensor,
                 margin: float, metric: str = "euclidean") -> torch.Tensor:
    """Your own triplet loss over a batch of (anchor, positive, negative) embeddings; returns a scalar."""
    # TODO (no nn.TripletMarginLoss or any other existing implementation)
    raise NotImplementedError


def train_triplet_epoch(backbone: nn.Module, optimizer, margin: float, metric: str) -> dict:
    """One epoch of triplet training of the backbone (no classifier); return e.g. {"loss": ...}."""
    # TODO (add the arguments your sampling needs)
    raise NotImplementedError

In [ ]:
def recall_at_k(emb: torch.Tensor, labels: torch.Tensor, ks=(5, 10, 50, 100),
                metric: str = "euclidean") -> dict:
    """Mean R(x_i, k) over all test images for each k (section 3.4 of the brief). Returns {k: mean recall}."""
    # TODO. A query must not count as its own neighbour. Check the Moodle FAQ on whether the class
    # size |y_i| should include the query itself.
    raise NotImplementedError


def mean_pos_neg_distance(emb: torch.Tensor, labels: torch.Tensor, metric: str = "euclidean") -> dict:
    """Mean distance over all valid (anchor, positive) pairs and all (anchor, negative) pairs.

    Returns {"mean_pos_dist": ..., "mean_neg_dist": ...}. An image paired with itself is not a valid pair.
    """
    # TODO
    raise NotImplementedError

In [ ]:
T4_CFG = {
    "margins": [],       # TODO: margin values to compare
    "metric": None,      # TODO: distance used for training and evaluation
    "epochs": None,      # TODO
    "batch_size": None,  # TODO
    "lr": None,          # TODO
}
check_cfg(T4_CFG)

for label_type in ("fine", "coarse"):
    train_labels = train_fine if label_type == "fine" else train_coarse
    test_labels = torch.as_tensor(test_fine if label_type == "fine" else test_coarse)
    for margin in T4_CFG["margins"]:
        run = f"task4_{label_type}_margin{margin}"
        set_seed(SEED)
        backbone = load_backbone().to(DEVICE)
        if RUN_TRAINING:
            # TODO: train `backbone` with your triplet sampling and triplet_loss, using train_labels.
            raise NotImplementedError
            save_checkpoint(backbone, f"{run}.pt", cfg={**T4_CFG, "margin": margin, "labels": label_type})
        else:
            load_checkpoint(backbone, f"{run}.pt")

        test_emb = extract_embeddings(backbone, test_loader)
        recalls = recall_at_k(test_emb, test_labels, metric=T4_CFG["metric"])
        RESULTS[run] = {
            **{f"recall@{k}": r for k, r in recalls.items()},
            **mean_pos_neg_distance(test_emb, test_labels, metric=T4_CFG["metric"]),
        }
        print(run, RESULTS[run])

In [ ]:
# TODO: tabulate or plot recall@k and the mean positive/negative distances against the margin, for the
# fine- and coarse-label experiments. Save the figures with savefig for the report.

## Task 5: Few-shot classification (12 marks)

Retrain the task 1 model on the task 5 split, whose training set drops 20 fine classes, so 20 test classes are never seen in training. Keep the task 1 architecture or change it (and explain why). Classify the test images with your k-NN over each provided support set, and report results for every support size and several values of k.

**Report:**
- The challenges of few-shot classification.
- How k, the distance metric and the support size affect performance.
- Any architecture changes, why you made them, and whether they beat the task 1 design.

In [ ]:
t5_train_images, t5_train_fine, t5_train_coarse = load_split(TASK5_TRAIN_PATH)
t5_test_images, t5_test_fine, t5_test_coarse = load_split(TASK5_TEST_PATH)
# size -> (images, fine_labels, coarse_labels); adapt if the support sets are stored differently.
t5_support = {size: load_split(path) for size, path in TASK5_SUPPORT_PATHS.items()}

# TODO: build the task 5 training (and validation) datasets. Check how the fine labels are numbered in
# this training split: the classifier's output size has to match the classes it is trained on.
t5_train_set = None  # TODO

In [ ]:
T5_CFG = {
    "num_classes": None,  # TODO: number of classes in the task 5 training split
    "epochs": None,       # TODO
    "batch_size": None,   # TODO
    "lr": None,           # TODO
}
check_cfg(T5_CFG)

set_seed(SEED)
t5_model = ImageClassifier(load_backbone(), ClassifierHead(EMB_DIM, T5_CFG["num_classes"])).to(DEVICE)

if RUN_TRAINING:
    # TODO: train on t5_train_set as in task 1 (or with your modified design).
    raise NotImplementedError
    save_checkpoint(t5_model, "task5_fewshot.pt", cfg=T5_CFG)
else:
    load_checkpoint(t5_model, "task5_fewshot.pt")

In [ ]:
T5_EVAL = {
    "ks": [],       # TODO: values of k to compare
    "metrics": [],  # TODO: distance metrics to compare
}
check_cfg(T5_EVAL)

# TODO: embed the task 5 test images with t5_model.backbone. Then, for each support-set size, metric
# and k: embed the support images, classify the test images with knn_predict, and store the top-1
# accuracy in RESULTS["task5"]. Plot accuracy against k (one line per support size) and save it.

## Task 6: Visualising embeddings (8 marks)

Project the test-set embeddings $f(x)$ from the task 2 model and from the coarse-label task 4 model with t-SNE, colouring each point by its coarse label. Any t-SNE implementation is allowed (e.g. scikit-learn's).

**Report:**
- How the embeddings cluster under each training scheme, and why.
- If you have task 4 models trained with different margins or sampling schemes, how they change the plots.

In [ ]:
from sklearn.manifold import TSNE  # allowed for task 6 only


def plot_embedding_2d(points: np.ndarray, coarse_labels: np.ndarray, title: str) -> plt.Figure:
    """Scatter plot of 2-D t-SNE points coloured by coarse label (20 classes)."""
    # TODO (Matplotlib's default colour cycle has only 10 colours)
    raise NotImplementedError


# TODO: rebuild the task 2 model and your coarse task 4 model(s), load them with load_checkpoint, extract
# test-set embeddings, project each with TSNE and plot them side by side; save with savefig. Use the same
# t-SNE settings and random seed for every model so the plots are comparable.

## Task 7: Checking generalisation (12 marks)

Does the fine-grained model implicitly cluster images of the same coarse class? Load the task 1 backbone (no retraining), embed the training and test sets, and predict each test image's coarse label with your k-NN over the training embeddings. Evaluate top-1 accuracy on $D_{test}$ and compare with the task 2 classifier.

**Report:**
- How k and the distance metric affect performance.
- How the k-NN compares with task 2: did the fine-grained model learn the coarse classes? Why is that good or bad?
- The challenges of generalisation.

In [ ]:
T7_EVAL = {
    "ks": [],       # TODO: values of k to compare
    "metrics": [],  # TODO: distance metrics to compare
}
check_cfg(T7_EVAL)

# TODO: rebuild the task 1 model and load "task1_fine.pt" (no retraining). Embed the training and test
# sets (deterministic preprocessing, shuffle=False), predict coarse labels with knn_predict using the
# training-set coarse labels, and store top-1 accuracy per k and metric in RESULTS["task7"].
# Compare with RESULTS["task2"].

## Task 8: SIFT-based classifiers (20 marks)

Extract SIFT features for all images and encode them with Bag of Words (BoW) or Fisher Vectors (FV). The encoding may differ between sub-tasks.

1. **Coarse- and fine-grained classification:** one SVM for fine labels and one for coarse labels, evaluated as in tasks 1 and 2. Compare with tasks 1 and 2.
2. **Few-shot classification:** fit K-means or the GMM on the task 5 training set, use it to encode the support and test sets, and classify with your k-NN using the support set as training data. Same protocol as task 5; compare with task 5.
3. **Generalisation:** a coarse-grained k-NN classifier, compared with task 7.

SIFT, SVMs, K-means and Fisher Vectors may come from libraries (e.g. scikit-image, scikit-learn). BoW, if used, and the k-NN must be your own (PyTorch or NumPy). This task is not evaluated on a held-out test set.

**Report:**
- Which encoding(s) you chose and why.
- How the SIFT, encoding and classifier hyper-parameters affected performance.
- How the SIFT-based models compare with the CNN models in each sub-task.

In [ ]:
# TODO: import the libraries you use for SIFT, SVMs, K-means / GMMs and (if used) Fisher Vectors, e.g.
# from skimage.feature import SIFT
# from sklearn.svm import SVC
# from sklearn.cluster import KMeans


def extract_sift(images: np.ndarray, **params) -> list:
    """SIFT descriptors: a list with one (n_i, 128) array per image.

    CIFAR images are 32x32, so some may yield few or no keypoints. Every image still needs an entry,
    and the encoders below must handle an empty one.
    """
    # TODO
    raise NotImplementedError


def fit_codebook(train_descriptors: list, n_words: int, **params):
    """BoW vocabulary: K-means (a library implementation is allowed) on training descriptors only."""
    # TODO
    raise NotImplementedError


def encode_bow(descriptors: list, codebook) -> np.ndarray:
    """Your own BoW encoding: one histogram over the visual words per image, shape (N, n_words)."""
    # TODO
    raise NotImplementedError


def fit_gmm(train_descriptors: list, n_components: int, **params):
    """FV only: a GMM fitted on training descriptors only."""
    # TODO, if you use Fisher Vectors
    raise NotImplementedError


def encode_fisher(descriptors: list, gmm) -> np.ndarray:
    """FV only: one Fisher Vector per image, shape (N, encoding_dim)."""
    # TODO, if you use Fisher Vectors
    raise NotImplementedError

In [ ]:
# 8.1 Coarse- and fine-grained classification with SVMs
# TODO: extract SIFT for the training and test sets; fit the codebook / GMM on the training descriptors;
# encode both sets; train one SVM on the fine labels and one on the coarse labels; store top-1 accuracy
# in RESULTS["task8_svm"] and compare with RESULTS["task1"] and RESULTS["task2"].

In [ ]:
# 8.2 Few-shot classification with k-NN
# TODO: fit the codebook / GMM on the task 5 training split; encode every support set and the task 5 test
# set with it; classify with knn_predict (convert with torch.from_numpy), using the support set as training
# data, for each support size and k as in task 5. Store in RESULTS["task8_fewshot"]; compare with RESULTS["task5"].

In [ ]:
# 8.3 Generalisation: coarse-grained k-NN
# TODO: classify the encoded test set by coarse label with knn_predict over the encoded training set, for
# the k values used in task 7. Store in RESULTS["task8_knn_coarse"] and compare with RESULTS["task7"].

## Results summary

In [ ]:
save_results()
for name, metrics in RESULTS.items():
    print(f"{name}: {metrics}")